# 04 - Forecasting Model
Melatih model baseline (SARIMA) dan model Machine Learning (XGBoost) untuk memprediksi inflasi.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error
import xgboost as xgb
import os

## 1. Load Feature Dataset

In [ ]:
df_path = '../data/features/feature_dataset.csv'
if os.path.exists(df_path):
    df = pd.read_csv(df_path, parse_dates=['date'])
    df = df.sort_values('date').dropna(subset=['inflation_yoy'])
    display(df.head())
else:
    print("❌ Feature dataset not found. Please run 03_feature_engineering.ipynb first.")
    df = pd.DataFrame()

## 2. Train/Test Split

In [ ]:
if not df.empty:
    train_size = int(len(df) * 0.8)
    train_df = df.iloc[:train_size]
    test_df = df.iloc[train_size:]
    print(f"Train range: {train_df['date'].min()} to {train_df['date'].max()} ({len(train_df)} rows)")
else:
    print("⚠️ Skipping split: No data.")
    train_df, test_df = pd.DataFrame(), pd.DataFrame()

## 3. Baseline Model (SARIMA)

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

if not train_df.empty:
    y_train = train_df['inflation_yoy']
    y_test = test_df['inflation_yoy']

    model_sarima = SARIMAX(y_train, order=(1, 0, 0))
    results_sarima = model_sarima.fit(disp=False)
    pred_sarima = results_sarima.forecast(steps=len(y_test))

    print(f"SARIMA MAE: {mean_absolute_error(y_test, pred_sarima):.4f}")
else:
    print("⚠️ Skipping SARIMA: No data.")

## 4. Machine Learning Model (XGBoost)

In [ ]:
if not train_df.empty:
    X_train = train_df.drop(columns=['date', 'pcpipch', 'inflation_yoy']).fillna(0)
    X_test = test_df.drop(columns=['date', 'pcpipch', 'inflation_yoy']).fillna(0)

    model_xgb = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100)
    model_xgb.fit(X_train, y_train)
    pred_xgb = model_xgb.predict(X_test)

    print(f"XGBoost MAE: {mean_absolute_error(y_test, pred_xgb):.4f}")
else:
    print("⚠️ Skipping XGBoost: No data.")

## 5. Comparison & Visualization

In [ ]:
if not train_df.empty:
    plt.figure(figsize=(12, 6))
    plt.plot(test_df['date'], y_test, label='Actual', color='black', linewidth=2)
    plt.plot(test_df['date'], pred_sarima, label='SARIMA', linestyle='--')
    plt.plot(test_df['date'], pred_xgb, label='XGBoost', linestyle=':')
    plt.title("Forecasting Comparison: Actual vs Predicted")
    plt.legend()
    plt.show()